In [1]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv('/content/Most-Recent-Cohorts-Institution.csv')

/tmp/ipykernel_790/2858648587.py:1: DtypeWarning: Columns (9,1537,1540,1542,1547,1561,1589,1601,1602,1606,1608,1614,1615,1619,1620,1621,1622,1623,1624,1625,1626,1627,1628,1629,1725,1726,1727,1728,1729,1743,1815,1816,1817,1818,1823,1824,1830,1831,1879,1880,1881,1882,1883,1884,1885,1886,1887,1888,1889,1890,1891,1892,1893,1894,1895,1896,1897,1898,1909,1910,1911,1912,1913,1957,1958,1959,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1983,1984,2376,2377,2403,2404,2495,2496,2497,2498,2499,2500,2501,2502,2503,2504,2505,2506,2507,2508,2509,2510,2511,2512,2513,2514,2515,2516,2517,2518,2519,2520,2521,2522,2523,2524,2525,2526,2527,2528,2529,2530,2958,3215,3231,3235,3236) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/Most-Recent-Cohorts-Institution.csv')


In [4]:
df[:5]
# Select Features
# MD_EARN_WNE_INC1_P11 (), COSTT4_A (COST), TUITIONFEE_IN, UGDS (# of Students), GRAD_DEBT_MDN,

,UNITID,OPEID,OPEID6,INSTNM,CITY,STABBR,ZIP,ACCREDAGENCY,INSTURL,NPCURL,...,MD_EARN_WNE_INC1_P11,MD_EARN_WNE_INC2_P11,MD_EARN_WNE_INC3_P11,MD_EARN_WNE_INDEP0_P11,MD_EARN_WNE_INDEP1_P11,MD_EARN_WNE_MALE0_P11,MD_EARN_WNE_MALE1_P11,SCORECARD_SECTOR,EARN_THR_STATE,EARN_THR_NAT
0,100654,100200.0,1002.0,Alabama A & M University,Normal,AL,35762,Southern Association of Colleges and Schools C...,www.aamu.edu/,www.aamu.edu/admissions-aid/tuition-fees/net-p...,...,36650.0,41070.0,47016.0,38892.0,41738.0,38167.0,40250.0,4,30168.0,32842
1,100663,105200.0,1052.0,University of Alabama at Birmingham,Birmingham,AL,35294-0110,Southern Association of Colleges and Schools C...,https://www.uab.edu/,https://tcc.ruffalonl.com/University of Alabam...,...,47182.0,51896.0,54368.0,50488.0,51505.0,46559.0,59181.0,4,30168.0,32842
2,100690,2503400.0,25034.0,Amridge University,Montgomery,AL,36117-3553,Southern Association of Colleges and Schools C...,https://www.amridgeuniversity.edu/,https://www2.amridgeuniversity.edu:9091/,...,35752.0,41007.0,NaN,NaN,38467.0,32654.0,49435.0,5,30168.0,32842
3,100706,105500.0,1055.0,University of Alabama in Huntsville,Huntsville,AL,35899,Southern Association of Colleges and Schools C...,www.uah.edu/,uah.clearcostcalculator.com/student/default/ne...,...,51208.0,62219.0,62577.0,55920.0,60221.0,47787.0,67454.0,4,30168.0,32842
4,100724,100500.0,1005.0,Alabama State University,Montgomery,AL,36104-0271,Southern Association of Colleges and Schools C...,www.alasu.edu/,tcc.ruffalonl.com/Alabama State University/Fre...,...,32844.0,36932.0,37966.0,34294.0,31797.0,32303.0,36964.0,4,30168.0,32842


In [37]:
# Select my features for matrix (Selections explained in medium post)
feature_cols = [
    'PREDDEG', 'HIGHDEG', 'REGION', 'LOCALE',
    'MD_EARN_WNE_INC1_P11', 'COSTT4_A', 'TUITIONFEE_IN', 'UGDS', 'GRAD_DEBT_MDN', 'DEBT_MDN', 'TUITFTE','ADM_RATE','CONTROL'
]
# Deal with N/As
#  'CONTROL'
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())
# Set the names of the colleges (INSTNM) as the index
df_features = df.set_index('INSTNM')[feature_cols]

In [38]:
# Check DF
df_features[:5]

,PREDDEG,HIGHDEG,REGION,LOCALE,MD_EARN_WNE_INC1_P11,COSTT4_A,TUITIONFEE_IN,UGDS,GRAD_DEBT_MDN,DEBT_MDN,TUITFTE,ADM_RATE
INSTNM,,,,,,,,,,,,
Alabama A & M University,3,4,5,12.0,36650.0,27153.0,10024.0,6124.0,31000.0,16600.0,7729.0,0.5795
University of Alabama at Birmingham,3,4,5,12.0,47182.0,28145.0,9098.0,11635.0,22300.0,15832.0,12322.0,0.8818
Amridge University,3,4,5,12.0,35752.0,25497.0,7590.0,241.0,32189.0,13385.0,14005.0,0.7793
University of Alabama in Huntsville,3,4,5,12.0,51208.0,27392.0,12132.0,6591.0,20705.0,13905.0,9420.0,0.6857
Alabama State University,3,4,5,12.0,32844.0,23586.0,11248.0,3477.0,31000.0,17500.0,9820.0,0.9755


In [49]:
# Normalize Data
df_min = df_features.min(axis=0)
df_max = df_features.max(axis=0)
df_norm = (df_features - df_min) / (df_max - df_min)
df_norm

,PREDDEG,HIGHDEG,REGION,LOCALE,MD_EARN_WNE_INC1_P11,COSTT4_A,TUITIONFEE_IN,UGDS,GRAD_DEBT_MDN,DEBT_MDN,TUITFTE,ADM_RATE
INSTNM,,,,,,,,,,,,
Alabama A & M University,0.75,1.0,0.555556,0.326087,0.210559,0.254904,0.139035,0.037533,0.701253,0.395919,0.032713,0.5795
University of Alabama at Birmingham,0.75,1.0,0.555556,0.326087,0.298762,0.266042,0.126191,0.071309,0.485039,0.375189,0.052153,0.8818
Amridge University,0.75,1.0,0.555556,0.326087,0.203038,0.236310,0.105275,0.001477,0.730802,0.309139,0.059276,0.7793
University of Alabama in Huntsville,0.75,1.0,0.555556,0.326087,0.332479,0.257587,0.168273,0.040395,0.445400,0.323175,0.039870,0.6857
Alabama State University,0.75,1.0,0.555556,0.326087,0.178684,0.214853,0.156012,0.021310,0.701253,0.420212,0.041563,0.9755
...,...,...,...,...,...,...,...,...,...,...,...,...
Valley College - Fairlawn - School of Nursing,0.00,0.0,0.333333,0.521739,0.122096,0.236310,0.165659,0.003310,0.166932,0.202062,0.044357,0.7793
Western Maricopa Education Center - Southwest Campus,0.00,0.0,0.666667,0.521739,0.201355,0.236310,0.165659,0.003310,0.264650,0.204276,0.044357,0.7793
Western Maricopa Education Center - Northeast Campus,0.00,0.0,0.666667,0.521739,0.201355,0.236310,0.165659,0.003310,0.264650,0.204276,0.044357,0.7793


In [41]:
import scipy.spatial.distance

In [50]:
# Function For getting Euclidean distance
def find_similar_colleges(target_college_name):
    if target_college_name not in df_norm.index:
        return "College not found."
    target_vector = df_norm.loc[target_college_name]
    # Calculate Euclidean distances to all other colleges
    distances = scipy.spatial.distance.cdist(df_norm, [target_vector], metric="euclidean")[:, 0]
    results = list(zip(df_norm.index, distances))
    top_results = sorted(results, key=lambda x: x[1])[0:11]
    print(f"\nTop 10 Colleges most similar to: {target_college_name}")
    print(f"{'-'*65}")
    for name, score in top_results:
        print(f"{name:<50} | Distance: {score:.4f}")

In [51]:
# Pick 3 Queries or Colleges
queries = [
    "Alabama A & M University",
    "University of Maryland-College Park",
    "Harvard University"
]

for college in queries:
    find_similar_colleges(college)


Top 10 Colleges most similar to: Alabama A & M University
-----------------------------------------------------------------
Alabama A & M University                           | Distance: 0.0000
Tennessee State University                         | Distance: 0.1654
Tougaloo College                                   | Distance: 0.1880
North Carolina A & T State University              | Distance: 0.2028
University of South Alabama                        | Distance: 0.2082
Hallmark University                                | Distance: 0.2099
Southern University Law Center                     | Distance: 0.2227
College of Charleston                              | Distance: 0.2262
Florida Institute of Technology-Online             | Distance: 0.2276
Amridge University                                 | Distance: 0.2278
Southern University at New Orleans                 | Distance: 0.2334

Top 10 Colleges most similar to: University of Maryland-College Park
-----------------------------------

## Euclidean Distance with only post grad economic varibales


In [44]:
feature_cols2 = [
    'PREDDEG', 'HIGHDEG',
    'MD_EARN_WNE_INC1_P11', 'GRAD_DEBT_MDN', 'DEBT_MDN'
]
df_features2 = df.set_index('INSTNM')[feature_cols2]

In [46]:
df_min1 = df_features2.min(axis=0)
df_max1 = df_features2.max(axis=0)
df_norm1 = (df_features2 - df_min1) / (df_max1 - df_min1)

In [48]:
# Function For getting Euclidean distance
def find_similar_colleges_new(target_college_name):
    if target_college_name not in df_norm1.index:
        return "College not found."
    target_vector = df_norm1.loc[target_college_name]
    # Calculate Euclidean distances to all other colleges
    distances = scipy.spatial.distance.cdist(df_norm1, [target_vector], metric="euclidean")[:, 0]
    results = list(zip(df_norm1.index, distances))
    top_results = sorted(results, key=lambda x: x[1])[0:11]
    print(f"\nTop 10 Colleges most similar to: {target_college_name}")
    print(f"{'-'*65}")
    for name, score in top_results:
        print(f"{name:<50} | Distance: {score:.4f}")

In [52]:
queries = [
    "Alabama A & M University",
    "University of Maryland-College Park",
    "Harvard University"
]

for college in queries:
    find_similar_colleges_new(college)


Top 10 Colleges most similar to: Alabama A & M University
-----------------------------------------------------------------
Alabama A & M University                           | Distance: 0.0000
Florida Memorial University                        | Distance: 0.0240
Huston-Tillotson University                        | Distance: 0.0356
Alabama State University                           | Distance: 0.0401
University of Phoenix-Arizona                      | Distance: 0.0470
University of Phoenix-California                   | Distance: 0.0470
Texas Southern University                          | Distance: 0.0546
Norfolk State University                           | Distance: 0.0557
Chowan University                                  | Distance: 0.0569
Southern University and A & M College              | Distance: 0.0591
Southern University Law Center                     | Distance: 0.0591

Top 10 Colleges most similar to: University of Maryland-College Park
-----------------------------------